In [7]:
#importing required python classes and packages
from bert_score import BERTScorer
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from datasets import Dataset
import torch
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [8]:
# Loading and initalizing GPT-4o tokenizer and model
gpt_tokenizer = AutoTokenizer.from_pretrained("model/gpt-4o")
gpt_model = AutoModelForCausalLM.from_pretrained("model/gpt-4o")
print("GPT Model Loaded")

GPT Model Loaded


In [9]:
#loading & displaying code review dataset
df = pd.read_csv("Dataset/code_review_data.csv",nrows=3000)
df

,patch,lang,prompt,responce,reward,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10
0,[KEEP]namespace DotNetNuke.UI.Containers\n[KEE...,none,"analyze the code and write a code review, if t...","Please use `String#IndexOf(String, StringCompa...",-7.26171875,NaN,NaN,NaN,NaN,NaN,NaN
1,[KEEP]func TestBlockchain_AccountState(t *test...,go,"analyze the code and write a code review, if t...","everything is fine, LGTM.",-5.38671875,NaN,NaN,NaN,NaN,NaN,NaN
2,[KEEP]def init_logger_experiment() -> None:\n[...,none,"analyze the code and write a code review, if t...",what is the relation between `init_logger_expe...,-8.828125,NaN,NaN,NaN,NaN,NaN,NaN
3,"[KEEP]require ""rails_helper""\n[KEEP] describe ...",rb,"analyze the code and write a code review, if t...",Put a comma after the last parameter of a mult...,-8.6875,NaN,NaN,NaN,NaN,NaN,NaN
4,[KEEP]func (j journalBlockServer) RemoveBlockR...,go,"analyze the code and write a code review, if t...","everything is fine, LGTM.",-6.06640625,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2995,[KEEP]def events_for_refund_transfer(\n[KEEP] ...,none,"analyze the code and write a code review, if t...",Should we rename the function to `events_for_s...,-7.7421875,NaN,NaN,NaN,NaN,NaN,NaN
2996,[KEEP]public class SslHandler extends ByteToMe...,none,"analyze the code and write a code review, if t...",remove empty line,-10.5,NaN,NaN,NaN,NaN,NaN,NaN
2997,[KEEP]class RaidenEventHandler:\n[KEEP] ...,none,"analyze the code and write a code review, if t...","For this file, the value of `given_block_ident...",-6.43359375,NaN,NaN,NaN,NaN,NaN,NaN
2998,[KEEP] * For any non-SNP sites in the input c...,java,"analyze the code and write a code review, if t...",as of 4.0.5.0\nrefinement:\n * </p>\n *\n *...,-6.3125,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
#intializing Bert Scorer
scorer = BERTScorer(model_type='bert-base-uncased', lang='en', rescale_with_baseline=True)
print("Bert Scorer Model Initialized")

Bert Scorer Model Initialized


In [11]:
#function to calculate bert score between true and predicted code review
def calculateBertScore():
    patch = df['patch'].ravel() #get code
    response = df['responce'].ravel()#get true review response 
    code = patch[159]
    prompt = code+'<|endoftext|>Comment:'
    inputs = gpt_tokenizer(prompt, return_tensors="pt", truncation=True, padding=True, max_length=1024)#generate tokens
    with torch.no_grad():
        #apply GPT model to predict code review
        outputs = gpt_model.generate(inputs.input_ids, max_length=1024, num_return_sequences=1, temperature=0.7, do_sample=True,
                                            pad_token_id=gpt_tokenizer.eos_token_id)
        review = gpt_tokenizer.decode(outputs[0], skip_special_tokens=True)
        #calculate bert score between true and predicted review
        s1, s2, s3 = scorer.score([response[1]], [review])
        score = s1.numpy()[0] + s3.numpy()[0]
        return score

In [12]:
#call bert score function to get BERT score
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token # Or choose another appropriate token
score = calculateBertScore()
print("GPT-4o Predicted F1 score = "+str(score))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


GPT-4o Predicted F1 score = 0.8807377


In [13]:
#function to generate review from given test code
def generateReview(test_code_file):
    with open(test_code_file, "rb") as file:
        code = file.read()
    file.close()
    code = code.decode()
    prompt = code+'<|endoftext|>Comment:'
    inputs = gpt_tokenizer(prompt, return_tensors="pt", truncation=True, padding=True, max_length=1024)#generate tokens
    with torch.no_grad():
        #apply GPT model to predict code review
        outputs = gpt_model.generate(inputs.input_ids, max_length=1024, num_return_sequences=1, temperature=0.7, do_sample=True,
                                            pad_token_id=gpt_tokenizer.eos_token_id)
        review = gpt_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return code, review    

In [14]:
#call this function to predict review from given test code
code, review = generateReview("TestCodes/50.txt")
#print("Submitted Code =====> "+str(code))
print("Generated Review =====> "+str(review))

Generated Review =====> def _mod_config(self, ll_addrs):
         bucket_directive = ['server_names_hash_bucket_size', '128']
 
         main = self.configurator.parser.parsed[root]
        for key, body in main:
        for block in main:
            key = block[0] if len(block) > 0 else None
            body = block[1] if len(block) > 1 else None

             if key == ['http']:
                 found_bucket = False
                for k, _ in body:
                    if k == bucket_directive[0]:
                for b in body:
                    if len(b) > 0 and b[0] == bucket_directive[0]:Comment: everything is fine, LGTM.
